# Lab3 · 端到端语音识别 (CTC + Bi-LSTM) — 填空版

本 notebook 一共 **6 个 TODO**, 都以 `# TODO N:` 开头, 占位符是 `___________`。
提示在注释里都给好了, 你只需要把对应的下划线替换成代码。

提交时请完整运行一遍, 保留所有输出。


# Lab3 · 端到端语音识别 (CTC + Bi-LSTM)

本实验在经典的 **AN4 数据集**上, 用一个 **双向 LSTM + CTC Loss** 的字符级模型, 实现一个最简单的端到端语音识别 (Automatic Speech Recognition, ASR)。

## 数据集说明

**AN4** (Census database) 由 CMU 在 1991 年录制, 是 SphinxTrain 的官方示例数据。内容是讲话人念出来的:
- 字母拼写: `R U B O U T`, `C I N D Y`
- 数字与日期: `ONE FIVE TWO THREE SIX`, `OCTOBER TWENTY THIRD NINETEEN SIXTY SEVEN`
- 控制指令: `YES`, `NO`, `GO`, `STOP`, `ENTER`, `ERASE`

| 划分 | 句子数 | 时长 (估) |
|------|--------|----------|
| train | 948 | ~50 min |
| test  | 130 | ~7 min |

## 你将要做的事

| 步骤 | 内容 |
|------|------|
| Part A | **看声音**: 加载 wav, 画波形, 画 Mel 频谱 |
| Part B | **训 CTC**: 加载 AN4, 定义模型, 训练 20 epoch, 看曲线 |
| Part C | **让它更好**: 加 SpecAugment + Speed Perturbation, 把测试 WER 从 ~47% 降到 ~36% |

## Stage 0 · 装环境 + 导入依赖

本次实验依赖 `torch`, `torchcodec`, `jiwer`, `matplotlib`。
第一次跑请先执行下面的安装 cell, 后面再跑就可以跳过它。

In [ ]:
import os, time, random
import torch
import torch.nn as nn
from torchcodec.decoders._audio_decoder import AudioDecoder
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from utils import (
    load_an4_split, TorchMelSpectrogram, text_to_indices, indices_to_text,
    ctc_greedy_decode, WerCerMeter, SpecAugment, expand_with_speed_perturb,
)

torch.manual_seed(42); random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

## Part A · 看声音

### Stage 1 · 读一段音频, 画时域波形

AN4 的音频是 16 kHz 单声道, 用 `torchcodec` 读进来得到一个形状 `(1, N_samples)` 的 tensor。我们先看其中一条录音的『波形』——即随时间变化的振幅。

In [ ]:
wav_path = './data/AN4/wav/an4_clstk/fash/cen5-fash-b.wav'   # 'P I T T S B U R G H'
decoder = AudioDecoder(wav_path)
samples = decoder.get_all_samples()
waveform = samples.data
sample_rate = samples.sample_rate
print('waveform shape:', waveform.shape, '| sample_rate:', sample_rate)
print('duration:', waveform.shape[1] / sample_rate, 'seconds')

plt.figure(figsize=(10, 2.5))
plt.plot(waveform[0].numpy(), linewidth=0.5)
plt.title('Waveform — P I T T S B U R G H'); plt.xlabel('sample index'); plt.ylabel('amplitude')
plt.tight_layout(); plt.show()

### Stage 2 · 计算 Mel 频谱, 把『声音』画成『图像』

波形信号维度很高 (1 秒 = 16000 个数字), 直接喂给神经网络效率很低, 且信息密度低。

标准做法是把波形先变成 **Mel 频谱图** (Mel-Spectrogram):

1. **分帧**: 25 ms 一帧, 帧间滑 10 ms (具体的样本点数要算: 采样率 × 时长)
2. **加窗 + FFT**: 每帧做一次傅里叶变换, 得到频谱
3. **Mel 滤波器组**: 把线性频率映射到 mel 频率轴 (模拟人耳: 低频分辨率高、高频分辨率低)

得到形状 `(n_mels, T)` 的张量, 其中 `T ≈ 时长 × (1秒 / 帧移)`。

In [ ]:
# TODO 1: 填写 MelSpectrogram 的 4 个参数 (参考课件 p.49-53)
# 提示：
#   sample_rate = AN4 的采样率 (单位 Hz) — 顶部数据集说明 / Stage 1 输出里都有
#   n_mels      = 80   (本 lab 约定的 mel 频带数, 是 ASR 业界标配)
#   n_fft       = 一帧的样本点数 = 采样率 × 窗长(秒)   ← 自己算: 25 ms 窗 @ 16000 Hz?
#   hop_length  = 帧间步长(样本点)= 采样率 × 帧移(秒) ← 自己算: 10 ms 帧移 @ 16000 Hz?
mel_spectrogram = TorchMelSpectrogram(
    sample_rate=16000,
    n_mels=80,
    n_fft=400,
    hop_length=160,
)
mel = mel_spectrogram(waveform)           # (1, 80, T)
print('mel shape:', mel.shape)

# log 之后再可视化, 因为人耳对响度是对数感知
log_mel = torch.log(mel + 1e-9)[0]        # (80, T)

plt.figure(figsize=(10, 3))
plt.imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='log mel energy')
plt.title('Log-Mel Spectrogram — P I T T S B U R G H')
plt.xlabel('time frame (10 ms each)'); plt.ylabel('mel bin (0 = low freq)')
plt.tight_layout(); plt.show()

> 🔬 **观察**: 横轴 ~450 帧 (≈ 4.5 秒); 纵轴 80 个 mel bin。每一列就是<strong>这一帧的 80 维特征</strong>, 也就是后面 Bi-LSTM 一个时间步的输入。
>
> ⚠️ **后续模型的输入约定是 `(T, 80)`** 而不是 `(80, T)` — 一个时间步是一行。我们在数据预处理时会做 `.transpose(0, 1)` 把维度调过来。

## Part B · 训练 CTC 模型

### Stage 3 · 加载 AN4 数据集

`utils.load_an4_split` 会读 `etc/an4_{train,test}.fileids` 和 `etc/an4_{train,test}.transcription`, 返回一个 list, 每个元素是 `(file_id, waveform, sample_rate, transcript_text)`。

In [ ]:
DATA_DIR = './data/AN4'

t0 = time.time()
train_raw = load_an4_split(DATA_DIR, 'train')
test_raw  = load_an4_split(DATA_DIR, 'test')
print(f'train: {len(train_raw)} clips, test: {len(test_raw)} clips, loaded in {time.time()-t0:.1f}s')

# 看一条样本
fid, wav, sr, txt = train_raw[10]
print('sample:', fid)
print('  waveform:', wav.shape, '  sr:', sr, '  text:', repr(txt))

# TODO 2: 在词表头部插入 CTC 的 blank token
# 提示：
#   - CTC 要求词表里有一个特殊的『空』符号, 模型用它表示『这一帧没说话』
#   - 我们约定 blank 的索引是 0, 这样 nn.CTCLoss(blank=0) 才认
#   - 字符串『<blank>』作为 token 名 (任意写法都行, 但 0 必须是 blank)
blank_id = 0               # blank 的索引位置 (应当是 0)
vocab = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ ')
vocab.insert(blank_id, '<blank>')  # 插入 blank token
print('vocab (', len(vocab), '):', vocab)

### Stage 4 · 数据预处理 (波形 → mel, 文本 → 索引)

对每条样本:
- 把 waveform 转成 `(T, 80)` 的 mel-spec
- 把 transcript 字符串转成索引列表, 例如 `'YES'` → `[25, 5, 19]`

In [ ]:
def preprocess(record):
    fid, wav, sr, txt = record
    mel = mel_spectrogram(wav).squeeze(0).transpose(0, 1)     # (T, 80)
    idx = text_to_indices(txt, vocab)                           # list[int]
    return mel, idx

train_pp = [preprocess(r) for r in train_raw]
test_pp  = [preprocess(r) for r in test_raw]

mel0, idx0 = train_pp[10]
print('preprocessed mel shape:', mel0.shape, '  idx:', idx0[:30], '...')
print('decoded back:', indices_to_text(idx0, vocab))

### Stage 5 · 定义 Dataset / DataLoader (含变长 padding)

一个 batch 里的样本时长不一样, 需要 **pad 到当前 batch 的最大长度**。`collate_fn` 同时返回 padded 张量和原始长度——CTC Loss 必须知道每个样本的真实长度。

In [ ]:
class AN4Dataset(Dataset):
    def __init__(self, data, spec_augment=None):
        """spec_augment: 可选, 一个把 (T,80) 转换成 (T,80) 的可调用对象 (只在训练阶段用)"""
        self.data = data
        self.aug = spec_augment

    def __len__(self): return len(self.data)

    def __getitem__(self, i):
        mel, idx = self.data[i]
        if self.aug is not None:
            mel = self.aug(mel)
        return mel, idx

def collate_fn(batch):
    mels = [b[0] for b in batch]
    idxs = [torch.tensor(b[1], dtype=torch.long) for b in batch]
    mels_padded = pad_sequence(mels, batch_first=True)            # (B, T_max, 80)
    idxs_padded = pad_sequence(idxs, batch_first=True)            # (B, U_max)
    in_lens     = torch.tensor([m.size(0) for m in mels], dtype=torch.long)
    tgt_lens    = torch.tensor([len(i)    for i in idxs], dtype=torch.long)
    return mels_padded, idxs_padded, in_lens, tgt_lens

### Stage 6 · 定义模型

模型结构 (对应课件 p.79):

```
Mel-spec (B, T, input_dim)
    │  Bi-LSTM × 2  (双向、多层)
    ▼
  (B, T, hidden_dim × 2)        ← bidirectional 把 hidden 拼成两倍
    │  Linear
    ▼
logits  (B, T, output_dim)
```

**`output_dim = len(vocab) = 28`**, 其中 index 0 是 CTC 的 blank。

In [ ]:
class CTC_ASR_Model(nn.Module):
    """端到端 CTC 语音识别模型. 输入 (B, T, input_dim), 输出 (B, T, output_dim)."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # TODO 3: 根据课件 p.79, 定义 self.lstm 和 self.fc
        # 提示:
        #   - p.79 的版本带 dropout, 我们这版**不要 dropout** —
        #     看下面 forward 就知道: 链路里没有 dropout, 你 __init__ 里也别加
        #   - bidirectional=True 会让 LSTM 输出维度翻倍, 所以 Linear 的输入也要翻倍
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc   = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        # 已给出: 过 LSTM 取 output, 再过 Linear — 不调用 dropout
        x, _ = self.lstm(x)
        x    = self.fc(x)
        return x


# 实例化模型 (已给出, 无需修改)
INPUT_DIM, HIDDEN = 80, 256
model = CTC_ASR_Model(INPUT_DIM, HIDDEN, len(vocab)).to(device)
n_param = sum(p.numel() for p in model.parameters())
print(f'model has {n_param/1e6:.2f} M parameters')
print(model)

### Stage 7 · 训练

CTC 训练的关键是把模型的 `(B, T, V+1)` logits 喂给一个特殊的 loss 函数 (见课件 p.63-67), 它会自己处理 T ≠ U 的对齐问题。这个 loss 对输入形状有特定要求, 写训练循环时要先把模型输出整成它要的样子。

In [ ]:
def train_one_epoch(model, loader, optimizer, ctc_loss):
    model.train()
    total_loss = 0.0; n_batches = 0
    for mels, idxs, in_lens, tgt_lens in loader:
        mels = mels.to(device); idxs = idxs.to(device)

        # 前向: model(mels) 输出 (B, T, V+1) 的 logits (已给出)
        out = model(mels)

        # TODO 4: 把 logits 转成 CTC Loss 要求的 log_probs (参考课件 p.65 / 官方 doc)
        # 提示:
        #   - CTCLoss 要的是 log-probability (不是普通 logits, 也不是普通 softmax 概率)
        #   - 形状要求是 (T, B, V+1), 而 out 现在是 (B, T, V+1) — 还得换维度
        #   - 这两步都是 torch.Tensor 上的方法
        log_probs = out.log_softmax(-1).transpose(0, 1)  # (T, B, V+1)

        # 计算 CTC 损失 (已给出)
        loss = ctc_loss(log_probs, idxs, in_lens, tgt_lens)

        # TODO 5: 反向传播 + 优化器更新 (经典 3 步, 跟 lab1/lab2 一样)
        # 提示: 想一下 lab1/lab2 训练循环里那 3 步分别是哪些方法?
        optimizer.zero_grad()     # 梯度清零
        loss.backward()           # 反向传播
        optimizer.step()          # 参数更新

        total_loss += loss.item(); n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, loader):
    """评测函数已给出, 无需修改。返回 (test_loss, test_CER, test_WER)。"""
    model.eval()
    meter = WerCerMeter()
    total_loss = 0.0; n_batches = 0
    for mels, idxs, in_lens, tgt_lens in loader:
        mels = mels.to(device); idxs_d = idxs.to(device)
        out = model(mels)
        log_probs = out.log_softmax(-1).transpose(0, 1)
        loss = ctc_loss_eval(log_probs, idxs_d, in_lens, tgt_lens)
        total_loss += loss.item(); n_batches += 1

        # 切到真实长度再 CTC 解码: padding 位置的预测不可信
        full = out.argmax(-1).cpu().tolist()
        truncated = [full[i][:int(in_lens[i])] for i in range(len(full))]
        decoded = ctc_greedy_decode(truncated, blank_index=0)
        hyps = [indices_to_text(d, vocab) for d in decoded]
        refs = [indices_to_text(idxs[i, :tgt_lens[i]].tolist(), vocab) for i in range(idxs.size(0))]
        meter.update(refs, hyps)

    cer, wer = meter.compute()
    return total_loss / max(n_batches, 1), cer, wer


# TODO 6: 选择损失函数和优化器
# 提示:
#   - 损失函数: 不是 lab1/lab2 用的 nn.CrossEntropyLoss — 那个要求输入和目标对齐 (T=U)
#               看课件 p.63-67, 时序对齐用的是哪个 loss? 用的时候要指定 blank 的索引
#               (我们约定 blank=0, 再加 zero_infinity=True 避免短输入炸 loss)
#   - 优化器:   lab1/lab2 用过的那个就行, 推荐 lr=2e-3
ctc_loss      = nn.CTCLoss(blank=0, zero_infinity=True)
ctc_loss_eval = ctc_loss              # 评测时直接复用同一个对象
optimizer     = torch.optim.Adam(model.parameters(), lr=2e-3)

In [ ]:
BATCH = 16
EPOCHS = 30            # 课堂里可以保持 30; 课后想多训也可以

# --- T0 档: 纯 AN4, 不加任何增强 ---
train_loader = DataLoader(AN4Dataset(train_pp), batch_size=BATCH, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(AN4Dataset(test_pp),  batch_size=8,    shuffle=False, collate_fn=collate_fn)

history = {'train_loss': [], 'test_loss': [], 'test_cer': [], 'test_wer': []}
best_wer = 1.0
os.makedirs('./checkpoints', exist_ok=True)

for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss = train_one_epoch(model, train_loader, optimizer, ctc_loss)
    te_loss, te_cer, te_wer = evaluate(model, test_loader)
    history['train_loss'].append(tr_loss); history['test_loss'].append(te_loss)
    history['test_cer'].append(te_cer);    history['test_wer'].append(te_wer)
    if te_wer < best_wer:
        best_wer = te_wer
        torch.save(model.state_dict(), './checkpoints/best.pth')
    print(f'ep {ep+1:2d}/{EPOCHS}  train_loss={tr_loss:.3f}  test_loss={te_loss:.3f}  test_CER={te_cer:.3f}  test_WER={te_wer:.3f}  ({time.time()-t0:.1f}s)')

print(f'\nbest test WER = {best_wer:.3f}  (checkpoint saved to ./checkpoints/best.pth)')

### Stage 8 · 画训练曲线

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
epochs_x = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_x, history['train_loss'], label='train')
axes[0].plot(epochs_x, history['test_loss'],  label='test')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('CTC loss'); axes[0].legend(); axes[0].set_title('Loss')
axes[1].plot(epochs_x, history['test_cer'], label='test CER')
axes[1].plot(epochs_x, history['test_wer'], label='test WER')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('error rate'); axes[1].legend(); axes[1].set_title('Test CER / WER')
plt.tight_layout(); plt.show()

### Stage 9 · 推理: 看几个具体例子

In [ ]:
model.load_state_dict(torch.load('./checkpoints/best.pth'))
model.eval()

def predict_one(idx):
    fid, wav, sr, txt = test_raw[idx]
    mel = mel_spectrogram(wav).squeeze(0).transpose(0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(mel)
    # 单条样本没有 padding, 直接对 T 个 argmax 做 CTC 解码即可
    pred = ctc_greedy_decode(out.argmax(-1).cpu().tolist(), blank_index=0)[0]
    return fid, txt, indices_to_text(pred, vocab)

for i in [0, 5, 12, 30, 60, 90]:
    fid, ref, hyp = predict_one(i)
    print(f'[{fid:24s}]  REF: {ref!r}')
    print(f'                          HYP: {hyp!r}\n')

## Part C · 让它更好

刚才 (T0) 的测试 WER 大概在 **47%** 左右, 远高于训练集 WER ~33%, 是明显的<strong>过拟合</strong>。我们用两个常用的 ASR 数据增强技术来缩小这个 gap:

### Stage 10 · SpecAugment + Speed Perturbation (T2 档)

- **Speed Perturbation**: 把每条音频以 0.9× / 1.0× / 1.1× 三个速度做 resample, 训练集瞬间变成 3 倍 (948 → 2844)。
- **SpecAugment**: 在 mel-spec 上随机遮一段频带和一段时间, 强制模型对局部缺失鲁棒。

> ⚠️ 这两个增强要 **组合使用**: 单独加 SpecAugment 但不扩数据, 在 20 epoch 内反而会变差 (模型见到更多扰动但训练步数没增加, 来不及学); 配合 3× 数据后才能稳定提升。

In [ ]:
# 1) 把 3 倍 speed-perturb 后的原始 (waveform) 列表先生成, 再过一次 mel
print('expanding training data with speed perturbation (3×)...')
train_raw_3x = expand_with_speed_perturb(train_raw, speeds=(0.9, 1.0, 1.1))
print(f'  {len(train_raw)}  →  {len(train_raw_3x)}')
train_pp_3x = [preprocess(r) for r in train_raw_3x]

# 2) Dataset 上挂 SpecAugment, 每次 __getitem__ 时实时生成新的遮挡
spec_aug = SpecAugment(freq_mask_param=15, time_mask_param=35)
train_loader_t2 = DataLoader(AN4Dataset(train_pp_3x, spec_augment=spec_aug),
                              batch_size=BATCH, shuffle=True, collate_fn=collate_fn)

# 3) 重新初始化模型 (从零训, 公平对比), 跑 20 个 epoch
model2 = CTC_ASR_Model(INPUT_DIM, HIDDEN, len(vocab)).to(device)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=2e-3)

history2 = {'train_loss': [], 'test_loss': [], 'test_cer': [], 'test_wer': []}
best_wer2 = 1.0
for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss = train_one_epoch(model2, train_loader_t2, optimizer2, ctc_loss)
    te_loss, te_cer, te_wer = evaluate(model2, test_loader)
    history2['train_loss'].append(tr_loss); history2['test_loss'].append(te_loss)
    history2['test_cer'].append(te_cer);    history2['test_wer'].append(te_wer)
    if te_wer < best_wer2:
        best_wer2 = te_wer
        torch.save(model2.state_dict(), './checkpoints/best_t2.pth')
    print(f'ep {ep+1:2d}/{EPOCHS}  loss={tr_loss:.3f}  test_CER={te_cer:.3f}  test_WER={te_wer:.3f}  ({time.time()-t0:.1f}s)')

print(f'\nT2 best test WER = {best_wer2:.3f}  (vs T0 = {best_wer:.3f})')

In [ ]:
# 对比 T0 vs T2
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(history['test_wer'])+1),  history['test_wer'],  label='T0: AN4 only', marker='o')
ax.plot(range(1, len(history2['test_wer'])+1), history2['test_wer'], label='T2: AN4×3 + SpecAugment', marker='s')
ax.set_xlabel('epoch'); ax.set_ylabel('test WER'); ax.set_title('T0 vs T2'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Stage 11 · 同一批样本: T0 vs T2 谁更准

用 Part C 训出来的 T2 模型, 跑一遍 Stage 9 的同 6 条 test 样本, 直接对比预测结果。

In [ ]:
# 把 T0 / T2 的最佳 checkpoint 都加载回来
model.load_state_dict(torch.load('./checkpoints/best.pth'));      model.eval()
model2.load_state_dict(torch.load('./checkpoints/best_t2.pth')); model2.eval()

def predict_with(m, idx):
    fid, wav, sr, _ = test_raw[idx]
    mel = mel_spectrogram(wav).squeeze(0).transpose(0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out = m(mel)
    pred = ctc_greedy_decode(out.argmax(-1).cpu().tolist(), blank_index=0)[0]
    return indices_to_text(pred, vocab)

for i in [0, 5, 12, 30, 60, 90]:
    fid, wav, sr, ref = test_raw[i]
    print(f'[{fid:24s}]  REF: {ref!r}')
    print(f'                          T0:  {predict_with(model,  i)!r}')
    print(f'                          T2:  {predict_with(model2, i)!r}\n')

## 补充信息
### 网络描述
本实验采用 CTC 语音识别结构，输入为 80 维 Mel 频谱特征，时间维长度记为 $T$ ，输入张量形状为
$(B,T,80)$ 。网络主体是 1 层双向 LSTM，隐藏层维度为 256，双向拼接后输出维度变为 512，再接一个全连接层映射到 28 个类别。28 类分别对应 1 个 CTC blank、26 个大写字母和 1 个空格符号。模型输出为每个时间步上的字符分类 logits，形状为 $(B,T,28)$ ，训练时使用 CTC Loss 处理输入和标签长度不一致的问题。
### 参数设置
Mel 频谱参数为：采样率 16000 Hz，窗长 25 ms，对应 $n\_fft = 400$；帧移 10 ms，对应 $hop\_length = 160$；mel 频带数 $n\_mels = 80$。模型参数为：输入维度 80，LSTM 隐藏层维度 256，双向 LSTM，输出维度 28，batch size 为 16，训练轮数为 30，优化器使用 Adam，学习率为 $2 \times 10^{-3}$。损失函数为 CTC Loss，blank 索引设为 0，并开启 `zero_infinity=True` 以避免短序列导致的异常梯度。
### 运行结果
训练过程中，模型的训练损失和测试损失整体呈下降趋势，说明网络能够逐步学习到语音到字符序列的映射关系。初始 T0 配置下测试集 WER 约为 70%，加入 Speed Perturbation 和 SpecAugment 后，T2 配置的测试 WER 可进一步降到约 47%，泛化性能明显提升。